<a href="https://colab.research.google.com/github/panarisivalentino-pixel/prediccion-rendimiento-agricola-argentina/blob/main/notebooks/04_analisis_agro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#   Exploracion de Datos del Dataset Agro
## Dataset de Produccion

### `df_produccion` — Estimaciones agrícolas (MAGyP)

| Columna | Tipo | Descripción |
|---|---|---|
| `cultivo` | texto | Cultivo relevado (soja, maíz, trigo, etc.). Algunos tienen variantes `1ra`/`2da`/`total` |
| `anio` | entero | Año calendario de inicio de la campaña |
| `campania` | texto | Campaña agrícola (ej: `2023/2024`), abarca dos años calendario |
| `provincia` | texto | Nombre de la provincia |
| `provincia_id` | decimal | Código INDEC de la provincia (convertir a entero) |
| `departamento` | texto | Nombre del departamento/partido dentro de la provincia |
| `departamento_id` | decimal | Código INDEC del departamento (convertir a entero) |
| `superficie_sembrada_ha` | entero | Hectáreas sembradas |
| `superficie_cosechada_ha` | entero | Hectáreas efectivamente cosechadas (puede ser menor a la sembrada por pérdidas) |
| `produccion_tm` | entero | Producción total en toneladas |
| `rendimiento_kgxha` | entero | Rendimiento: kilos producidos por hectárea cosechada |

---


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

nombre_buscado = "Estimacion_Agricola Crudo.xlsx"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

 No se encontró el archivo con ese nombre exacto. Revisa el paso 3.


In [ ]:
import pandas as pd

# Ruta exacta donde se encontró el archivo
ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/Estimacion_Agricola Crudo.xlsx'

# Cargar el archivo Excel
df_produccion = pd.read_excel(ruta)

# Visualizar las primeras filas
df_produccion.head()

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha
0,ajo,1969,1969/1970,Buenos Aires,6.0,25 de Mayo,6854.0,3,3,10,3333
1,ajo,1969,1969/1970,Buenos Aires,6.0,Adolfo Gonzales Chaves,6014.0,15,15,82,5467
2,ajo,1969,1969/1970,Buenos Aires,6.0,Almirante Brown,6028.0,2,2,8,4000
3,ajo,1969,1969/1970,Buenos Aires,6.0,Balcarce,6063.0,450,450,2025,4500
4,ajo,1969,1969/1970,Buenos Aires,6.0,Cañuelas,6134.0,2,2,7,3500


In [ ]:
df_produccion.shape

(160499, 11)

El dataset tiene 160499 filas y 11 columnas

In [ ]:
df_produccion.dtypes

,0
cultivo,object
anio,int64
campania,object
provincia,object
provincia_id,float64
departamento,object
departamento_id,float64
superficie_sembrada_ha,int64
superficie_cosechada_ha,int64
produccion_tm,int64


`provincia_id` y `departamento_id` estan como `float64`(con decimales) en vez de `int64` (numeros enteros), en la limpieza hay que modificarlo para evitar errores en el futuro.

In [ ]:
df_produccion['cultivo'].unique()

array(['ajo', 'algodón', 'alpiste', 'arroz', 'arveja', 'avena', 'banana',
       'caña de azúcar', 'cártamo', 'cebada cervecera',
       'cebada forrajera', 'cebada total', 'cebolla total', 'centeno',
       'colza', 'garbanzo', 'girasol', 'jojoba', 'lenteja', 'limón',
       'lino', 'maíz', 'mandarina', 'maní', 'mijo', 'naranja',
       'papa total', 'pomelo', 'poroto alubia', 'poroto negro',
       'poroto otros', 'poroto total', 'soja 1ra', 'soja 2da',
       'soja total', 'sorgo', 'té', 'trigo candeal', 'trigo total',
       'tung', 'yerba mate'], dtype=object)

#### cultivos "repetidos"

`soja 1ra` / `soja 2da` / `soja total` → distintas fechas de siembra en la misma campaña.
`total` = suma de ambas.
`trigo candeal` / `trigo total`, `cebada cervecera` / `cebada forrajera` / `cebada total` → distintas variedades comerciales. `total` = suma de todas.

**Regla:** usar siempre la variante `"<cultivo> total"` para evitar contar producción dos veces.

In [ ]:
df_produccion['provincia'].unique()

array(['Buenos Aires', 'Entre Ríos', 'Formosa', 'Jujuy', 'La Rioja',
       'Mendoza', 'Catamarca', 'Misiones', 'Chaco', 'Neuquén',
       'Río Negro', 'Salta', 'Chubut', 'San Juan', 'Córdoba', 'San Luis',
       'Santa Fe', 'Santiago del Estero', 'Tucumán', 'Santa Cruz',
       'Corrientes', 'La Pampa', nan, 'Tierra del Fuego'], dtype=object)

El dataset tiene las 23 provincias de la Argentina y una con valor nulo

In [ ]:
df_produccion.isnull().sum()

,0
cultivo,0
anio,0
campania,0
provincia,8
provincia_id,8
departamento,8
departamento_id,8
superficie_sembrada_ha,0
superficie_cosechada_ha,0
produccion_tm,0


Tiene 8 valores nulos en 4 columnas: `provincia`, `provincia_id`, `departamento` y `departamento_id`